# F1 Card Battle — figure in vinile dei piloti

Trasforma le foto libere dei piloti (Wikimedia Commons, elenco in `src/data/portraits.json`) in **figure in vinile da collezione**
(testa grande, corpo piccolo, tuta con i colori della scuderia) che **somigliano al pilota**.

Usa **InstantID**: legge il volto dalla foto (lineamenti, forma del viso) e lo fa disegnare a Stable Diffusion XL nello stile scelto.

**Come si usa**
1. Menu **Runtime → Cambia tipo di runtime → GPU T4** → Salva.
2. Menu **Runtime → Esegui tutto**. La prima volta scarica circa 10 GB di modelli (5–10 minuti).
3. La cella **Prova** mostra alcuni piloti con 3 impostazioni diverse: scegli quella che ti piace e scrivila nelle impostazioni (cella 1).
4. Metti `TUTTI = True` nella cella 1 ed esegui di nuovo **Tutti i piloti**: alla fine scarica `figure.zip`.

Con `SALVA_SU_DRIVE = True` le figure finiscono anche su Google Drive: se Colab si disconnette, rieseguendo riparte da dove era arrivato.

In [ ]:
#@title 1 · Impostazioni
DA_ANNO = 2010            # piloti che hanno corso dal 2010 in poi (metti 1950 per tutti quelli con foto)
PROVA = ["lewis-hamilton", "fernando-alonso", "lando-norris", "oscar-piastri", "george-russell", "kimi-raikkonen"]
TUTTI = False             # True = genera tutti i piloti scelti da DA_ANNO
SALVA_SU_DRIVE = False    # True = salva anche su Google Drive (MyDrive/F1-figure) e riprende da lì

# impostazioni dello stile (la cella "Prova" ne confronta tre)
SOMIGLIANZA = 0.8         # quanto conta il volto della foto (0.5 = più stile, 1.0 = più somiglianza)
POSA = 0.6                # quanto si segue la posizione di occhi/naso/bocca
PASSI = 30                # qualità (più alto = più lento)
GUIDA = 5.0               # quanto si segue il testo del prompt
SEME = 7                  # cambia il numero per un'altra variante
MODELLO_BASE = "wangqixun/YamerMIX_v8"   # oppure "stabilityai/stable-diffusion-xl-base-1.0"

PROMPT = ("funko pop style vinyl toy figure of a {who}, chibi, oversized head, small body, "
          "big glossy eyes, wearing a {suit} racing suit, glossy vinyl plastic, 3d render, "
          "product photo, soft studio lighting, plain white background, centered")
NEGATIVO = ("text, logo, letters, watermark, box, packaging, realistic photo, realistic skin, "
            "lowres, blurry, deformed, extra limbs, extra fingers, cropped head")

In [ ]:
#@title 2 · Installazione (2–3 minuti)
import torch
assert torch.cuda.is_available(), "Nessuna GPU: Runtime → Cambia tipo di runtime → GPU T4, poi riesegui"
!pip install -q -U diffusers transformers accelerate insightface onnxruntime "rembg[cpu]"
# una sola copia di OpenCV (rembg e insightface ne portano altre: insieme si rompono a vicenda)
!pip uninstall -q -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
!pip install -q "opencv-python-headless>=4.8,<5"
!rm -rf InstantID && git clone -q --depth 1 https://github.com/instantX-research/InstantID
# le versioni recenti di diffusers hanno spostato MultiControlNetModel
!sed -i 's/^from diffusers.pipelines.controlnet.multicontrolnet import MultiControlNetModel/from diffusers.models.controlnets.multicontrolnet import MultiControlNetModel/' InstantID/pipeline_stable_diffusion_xl_instantid.py
# riconoscimento del volto (insightface antelopev2)
!mkdir -p models && [ -d models/antelopev2 ] || (wget -q https://github.com/deepinsight/insightface/releases/download/v0.7/antelopev2.zip && unzip -q -o antelopev2.zip -d models && rm antelopev2.zip)
!ls models/antelopev2

In [ ]:
#@title 3 · Piloti, foto e colori delle scuderie
import json, re, colorsys, requests
RAW = "https://raw.githubusercontent.com/J0joFra/F1-Card-Battle/main/"
cards = requests.get(RAW + "src/data/cards.json", timeout=60).json()["cards"]
portraits = requests.get(RAW + "src/data/portraits.json", timeout=60).json()
colors_ts = requests.get(RAW + "src/game/teamColors.ts", timeout=60).text
TEAM_HEX = dict(re.findall(r"'?([a-z0-9-]+)'?: '(#[0-9a-f]{6})'", colors_ts))

# come descrivere la tuta delle scuderie più recenti; le altre prendono il nome del colore
SUITS = {
    "ferrari": "red Ferrari", "mclaren": "papaya orange McLaren", "mercedes": "black and teal Mercedes",
    "red-bull": "dark navy blue Red Bull", "williams": "dark blue Williams", "aston-martin": "dark green Aston Martin",
    "alpine": "blue and pink Alpine", "haas": "white and red Haas", "racing-bulls": "white and blue",
    "rb": "white and blue", "alphatauri": "white and navy", "toro-rosso": "dark blue and red",
    "audi": "silver and black Audi", "cadillac": "black and gold", "kick-sauber": "black and neon green",
    "alfa-romeo": "red and white", "sauber": "white and red Sauber", "renault": "yellow and black Renault",
    "lotus-f1": "black and gold Lotus", "racing-point": "pink", "force-india": "black and orange",
    "manor": "red and black", "marussia": "red and black", "caterham": "green and yellow", "hrt": "white and red",
    "virgin": "black and red", "brawn": "white and neon yellow", "toyota": "white and red", "bmw-sauber": "white and blue",
}
def color_name(hexc):
    r, g, b = (int(hexc[i:i + 2], 16) / 255 for i in (1, 3, 5))
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    if s < 0.15: return "white" if l > 0.8 else "black" if l < 0.2 else "silver"
    names = [(15, "red"), (40, "orange"), (65, "yellow"), (160, "green"), (195, "teal"), (255, "blue"), (290, "purple"), (335, "pink"), (360, "red")]
    hue = next(n for lim, n in names if h * 360 <= lim)
    return ("dark " if l < 0.3 else "") + hue

def last_year(period):
    return max(int(y) for y in re.findall(r"\d{4}", str(period)))

drivers = {}   # id -> dati del pilota con la scuderia più recente
for c in cards:
    if c["kind"] != "team": continue
    d, y = c["driver"]["id"], last_year(c["period"])
    if d not in drivers or drivers[d]["year"] < y:
        team = c["team"]["id"]
        suit = SUITS.get(team) or color_name(TEAM_HEX.get(team, "#888888"))
        drivers[d] = {"name": c["driver"]["name"], "year": y, "team": c["team"]["name"], "suit": suit,
                      "female": bool(c["driver"].get("female"))}
# piloti degli anni d'oro senza carte squadra (pionieri/classici): tuta generica
for c in cards:
    d = c["driver"]["id"]
    if d not in drivers:
        drivers[d] = {"name": c["driver"]["name"], "year": c.get("firstYear") or 1950, "team": "", "suit": "white vintage",
                      "female": bool(c["driver"].get("female"))}

scelti = sorted(d for d, v in drivers.items() if d in portraits and v["year"] >= DA_ANNO)
print(f"{len(portraits)} foto, {len(scelti)} piloti dal {DA_ANNO} in poi")
print("esempio:", drivers.get("lewis-hamilton"))

In [ ]:
#@title 4 · Modelli (la prima volta 5–10 minuti)
import sys, cv2, numpy as np
sys.path.insert(0, "InstantID")
from huggingface_hub import hf_hub_download
from diffusers.models import ControlNetModel
from insightface.app import FaceAnalysis
from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline, draw_kps
from rembg import new_session, remove

for f in ["ControlNetModel/config.json", "ControlNetModel/diffusion_pytorch_model.safetensors", "ip-adapter.bin"]:
    hf_hub_download("InstantX/InstantID", f, local_dir="checkpoints")
faces = FaceAnalysis(name="antelopev2", root=".", providers=["CPUExecutionProvider"])
faces.prepare(ctx_id=-1, det_size=(640, 640))
controlnet = ControlNetModel.from_pretrained("checkpoints/ControlNetModel", torch_dtype=torch.float16)
pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(MODELLO_BASE, controlnet=controlnet, torch_dtype=torch.float16).to("cuda")
pipe.load_ip_adapter_instantid("checkpoints/ip-adapter.bin")
pipe.set_progress_bar_config(disable=True)
cutout = new_session("u2net")
print("pronto")

In [ ]:
#@title 5 · Funzioni
import io, os, time
from PIL import Image
http = requests.Session()
http.headers["User-Agent"] = "F1CardBattle/1.0 (https://github.com/J0joFra/F1-Card-Battle)"  # richiesto da Wikimedia
SIZE = 1024

def photo_of(driver_id):
    thumb = portraits[driver_id]["thumb"]
    url = re.sub(r"/(\d+)px-", "/960px-", thumb) if "/thumb/" in thumb else thumb
    res = http.get(url, timeout=30)
    if 400 <= res.status_code < 500:  # misura non disponibile: si usa la miniatura nota
        res = http.get(thumb, timeout=30)
    res.raise_for_status()
    return Image.open(io.BytesIO(res.content)).convert("RGB")

def face_of(img):
    found = faces.get(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR))
    if not found:
        return None
    return max(found, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))

def framed_kps(kps):
    # occhi, naso e bocca spostati dove sta la testa grande della figura: centrata, nella parte alta
    kps = np.asarray(kps, float)
    eyes = np.linalg.norm(kps[1] - kps[0])
    kps = (kps - kps.mean(0)) * (SIZE * 0.17 / max(eyes, 1))
    return kps + np.array([SIZE * 0.5, SIZE * 0.40])

def figure(driver_id, ip=None, cn=None, seed=None, face=None):
    d = drivers[driver_id]
    face = face_of(photo_of(driver_id)) if face is None else face
    if face is None:
        raise ValueError("nessun volto nella foto")
    kps_img = draw_kps(Image.new("RGB", (SIZE, SIZE)), framed_kps(face.kps))
    ip = SOMIGLIANZA if ip is None else ip
    pipe.set_ip_adapter_scale(ip)
    who = ("woman" if d["female"] else "man") + " racing driver"
    img = pipe(PROMPT.format(who=who, suit=d["suit"]), negative_prompt=NEGATIVO,
               image_embeds=face.embedding, image=kps_img,
               controlnet_conditioning_scale=POSA if cn is None else cn, ip_adapter_scale=ip,
               num_inference_steps=PASSI, guidance_scale=GUIDA, height=SIZE, width=SIZE,
               generator=torch.Generator("cuda").manual_seed(SEME if seed is None else seed)).images[0]
    return img

def avatar(img):
    # sfondo trasparente e ritaglio quadrato attorno alla figura, 256 px come gli avatar dell'app
    mask = remove(img, session=cutout, only_mask=True).convert("L")
    out = img.convert("RGBA"); out.putalpha(mask)
    box = mask.point(lambda v: 255 if v > 40 else 0).getbbox()
    if box:
        x0, y0, x1, y1 = box
        side = max(x1 - x0, y1 - y0) + 24
        cx, cy = (x0 + x1) // 2, (y0 + y1) // 2
        out = out.crop((cx - side // 2, cy - side // 2, cx + side // 2, cy + side // 2))
    return out.resize((256, 256), Image.LANCZOS)

def grid(rows, cell=256):
    sheet = Image.new("RGB", (cell * len(rows[0]), cell * len(rows)), "#e9eaee")
    for r, row in enumerate(rows):
        for c, im in enumerate(row):
            if im is None: continue
            im = im.convert("RGBA").resize((cell, cell))
            sheet.paste(im, (c * cell, r * cell), im)
    return sheet

## Prova
Per ogni pilota: la foto, poi 3 varianti (somiglianza / posa): **A** = 0.6 / 0.5 · **B** = 0.8 / 0.6 · **C** = 0.95 / 0.8.
Scegli la colonna che preferisci e scrivi i suoi valori in `SOMIGLIANZA` e `POSA` (cella 1).

In [ ]:
#@title 6 · Prova su pochi piloti
from IPython.display import display
VARIANTI = [(0.6, 0.5), (0.8, 0.6), (0.95, 0.8)]
rows = []
for d in PROVA:
    t = time.time()
    try:
        ph = photo_of(d); f = face_of(ph)
        if f is None: raise ValueError("nessun volto nella foto")
        side = min(ph.size); thumb = ph.crop(((ph.width - side) // 2, 0, (ph.width + side) // 2, side))
        rows.append([thumb] + [avatar(figure(d, ip, cn, face=f)) for ip, cn in VARIANTI])
        print(f"{d}: {time.time() - t:.0f}s")
    except Exception as e:
        print(f"{d}: {e}")
display(grid(rows))

## Tutti i piloti
Con `TUTTI = True` (cella 1). Circa 15–20 secondi a pilota sulla T4. Alla fine scarica `figure.zip`.

In [ ]:
#@title 7 · Tutti i piloti
import shutil
from google.colab import files
if not TUTTI:
    print("Metti TUTTI = True nella cella 1 e riesegui questa cella")
else:
    OUT = "figure"
    if SALVA_SU_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/F1-figure"
    os.makedirs(OUT, exist_ok=True)
    falliti, t0 = [], time.time()
    for i, d in enumerate(scelti):
        out = os.path.join(OUT, f"{d}.webp")
        if os.path.exists(out): continue   # già fatto (ripresa)
        try:
            avatar(figure(d)).save(out, "WEBP", quality=85, method=6)
        except Exception as e:  # una foto che non va non ferma le altre
            falliti.append(f"{d}: {e}")
        if i % 10 == 0:
            print(f"[{i + 1}/{len(scelti)}] {d} · {(time.time() - t0) / 60:.0f} min")
        time.sleep(0.2)
    print("Non riusciti:", *falliti, sep="\n  ")
    shutil.make_archive("figure", "zip", OUT)
    files.download("figure.zip")